# HW2: Spacing errors and subword segmentation in Korean PLMs (Colab, T4)

Run the cells **in order**. Results are written to Google Drive, so a disconnected session can resume:
finished runs are skipped automatically.

1. `Runtime → Change runtime type → T4 GPU`
2. Set `REPO_URL` in cell 2 to your GitHub repository (or upload the code zip as described there).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 2: code + Drive
REPO_URL = "https://github.com/<your-id>/korean-spacing-robustness"   # <- edit
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/hw2_results'
import os; os.makedirs(RESULTS, exist_ok=True)
%cd /content
!rm -rf repo && git clone --depth 1 $REPO_URL repo
# Without GitHub: upload hw2_code.zip in the Files panel, then run:  !unzip -o -q /content/hw2_code.zip -d /content/repo
%cd /content/repo
!pip install -q -r requirements.txt
!ln -sfn $RESULTS results

In [ ]:
# Cell 3: evaluation sets (already in data/; rebuilding must give identical files)
!md5sum data/setA_k1.jsonl data/setB_k1.jsonl data/setB_k2.jsonl
!bash code/make_sets.sh && md5sum data/setA_k1.jsonl data/setB_k1.jsonl data/setB_k2.jsonl

In [ ]:
# Cell 4: settings + PILOT (first main run: KLUE-BERT, seed 0; skipped if it already exists)
import os
SETS = "data/setA_k1.jsonl data/setB_k1.jsonl data/setB_k2.jsonl"
if not os.path.exists("results/bert-base_s0/train_log.json"):
    !python code/run_experiment.py --model klue/bert-base --seed 0 --sets $SETS --outdir results

In [ ]:
# Cell 5: main runs (3 models x 3 seeds). Finished runs are skipped. A run whose clean dev
# accuracy is at chance level (< 40%) has diverged; it is kept on disk but excluded, and the
# next seed (3, 4, ...) is run until each model has 3 valid runs.
import os, json, subprocess
MODELS = ["klue/bert-base", "bert-base-multilingual-cased", "xlm-roberta-base"]
for m in MODELS:
    tag0, seed, valid = m.split('/')[-1], 0, 0
    while valid < 3:
        log = f"results/{tag0}_s{seed}/train_log.json"
        if not os.path.exists(log):
            subprocess.run(f"python code/run_experiment.py --model {m} --seed {seed} --sets {SETS} --outdir results", shell=True, check=True)
        acc = json.load(open(log))["clean_dev_acc"]
        print(f"{tag0}_s{seed}: clean dev acc {acc:.3f}" + ("  -> diverged, excluded" if acc < 0.40 else ""))
        valid += acc >= 0.40
        seed += 1

In [ ]:
# Cell 6: segmentation change per tokenizer (CPU is enough)
!mkdir -p results/tokdiff
for name in ["setA_k1", "setB_k1", "setB_k2"]:
    !python code/token_diff.py --input data/{name}.jsonl --models klue/bert-base bert-base-multilingual-cased xlm-roberta-base --out results/tokdiff/{name}.csv

In [ ]:
# Cell 7: tables (+ dependent-noun check)
!python code/analyze.py --results results --sets data --tokdiff results/tokdiff --out results/tables
!python code/dep_tokens.py --set data/setA_k1.jsonl --models klue/bert-base bert-base-multilingual-cased xlm-roberta-base --preds results --out results/tables/table6_dep_tokens
!pip freeze > results/env_freeze.txt